# Stage 1: Case Study Dataset for Task 1 - 5
A common challenge for a car dealership when purchasing a used car at an auto auction is assessing the risk that the vehicle may have serious issues that prevent it from being sold to customers. The auto community calls these unfortunate purchases "kicks".

Kicked cars often result from tampered odometers, mechanical issues the dealer is unable to address, issues with obtaining the vehicle title from the seller, or other unforeseen problems. Kick cars can be very costly for dealers, including transportation costs, throwaway repair work, and market losses from reselling the vehicle.

A dealership manager would like to determine which cars are at higher risk of being kicked. This will provide real value to the dealership by offering the best possible inventory selection to its customers. They have been collecting data for many years and have also manually labelled it.

## Task 1: Data preparation for modelling (3.5 marks)
1. The dataset may include irrelevant and redundant variables to the underlying ML task. What variables did you include in the modelling, and what were their roles and measurement level set? Justify your choice.
2. Did you have to fix any data quality problems, including data imputation? Detail them.
3. Report the proportion of values of the target variable for the dataset before and after the pre-processing.

In [134]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split # TODO: do we need it?

FILE = "kick_1.csv"
# read the dataset
df = pd.read_csv(FILE)

def explore(df):
    skip_cols = ["PurchaseID", "PurchaseDate", "PurchaseTimestamp", "WarrantyCost"]
    for col in df.columns:
        if col in skip_cols:
            continue
        print(col, df[col].dtype, df[col].unique())

def normalize_str_column(df, col_name):
    df[col_name] = df[col_name].replace(["NOT AVAIL", "?", np.nan], "N/A").astype(str).str.upper()

def normalize_numeric_column(df, col_name):
    df[col_name] = df[col_name].fillna(df[col_name].mean())

def normalize_bool_column(df, col_name):
    df[col_name] = (
        df[col_name]
        .astype(str)
        .str.strip()
        .str.upper()
        .replace(["?", "NAN", "NONE", "", np.nan], pd.NA)
        .map({
            "YES": True,
            "Y": True,
            "NO": False,
            "N": False,
            "0": False,
            "0.0": False,
            "1": True,
            "1.0": True
        })
    )

def convert_string_to_int_na_fill_mean(df, col_name):
    df[col_name] = pd.to_numeric(df[col_name], errors="coerce")
    df[col_name] = df[col_name].fillna(df[col_name].mean()).round().astype("Int64") 

def convert_string_to_float_na_fill_mean(df, col_name):
    df[col_name] = pd.to_numeric(df[col_name], errors="coerce")
    df[col_name] = df[col_name].fillna(df[col_name].mean())
# Explore data:
#explore(df)
df.head()
# Drop irrelevant and repeated columns:
df.drop(['PurchaseID', 'PurchaseDate', 'PurchaseTimestamp', 'WheelTypeID', 'TopThreeAmericanName'], axis=1, inplace=True)

df["VehYear"] = pd.to_numeric(df["VehYear"], errors="coerce").astype("Int16")
df["VehOdo"] = pd.to_numeric(df["VehOdo"], errors="coerce").astype("Int32")
normalize_str_column(df, 'Auction')
normalize_str_column(df, 'Make')
normalize_str_column(df, 'Color')
normalize_str_column(df, 'Transmission')
normalize_str_column(df, 'WheelType')
normalize_str_column(df, 'Nationality')
normalize_numeric_column(df, 'WarrantyCost')
normalize_bool_column(df, 'PRIMEUNIT')
normalize_str_column(df, 'AUCGUART')
normalize_bool_column(df, 'ForSale')
normalize_bool_column(df, 'IsOnlineSale')
normalize_bool_column(df, 'IsBadBuy')
convert_string_to_int_na_fill_mean(df, 'MMRAcquisitionAuctionAveragePrice')
convert_string_to_int_na_fill_mean(df, 'MMRAcquisitionAuctionCleanPrice')
convert_string_to_int_na_fill_mean(df, 'MMRAcquisitionRetailAveragePrice')
convert_string_to_int_na_fill_mean(df, 'MMRAcquisitonRetailCleanPrice')
convert_string_to_int_na_fill_mean(df, 'MMRCurrentAuctionAveragePrice')
convert_string_to_int_na_fill_mean(df, 'MMRCurrentAuctionCleanPrice')
convert_string_to_int_na_fill_mean(df, 'MMRCurrentRetailAveragePrice')
convert_string_to_int_na_fill_mean(df, 'MMRCurrentRetailCleanPrice')
convert_string_to_float_na_fill_mean(df, 'MMRCurrentRetailRatio')

# Explore again:
#print('\n=== AFTER PROCESSING==')
explore(df)

Y = df["IsBadBuy"]
# pretty-print df:
df.head()


/tmp/ipykernel_20672/191566219.py:7: DtypeWarning: Columns (27) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(FILE)


Auction object ['OTHER' 'MANHEIM' 'N/A' 'ADESA']
VehYear Int16 <IntegerArray>
[2008, 2007, 2004, 2006, 2005, 2003, 2009, 2001, 2002, <NA>, 2010]
Length: 11, dtype: Int16
Make object ['DODGE' 'CHRYSLER' 'CHEVROLET' 'PONTIAC' 'MITSUBISHI' 'MAZDA' 'SUZUKI'
 'KIA' 'SATURN' 'TOYOTA' 'FORD' 'BUICK' 'JEEP' 'NISSAN' 'INFINITI'
 'HYUNDAI' 'GMC' 'VOLKSWAGEN' 'HONDA' 'MERCURY' 'OLDSMOBILE' 'ACURA'
 'CADILLAC' 'ISUZU' 'LINCOLN' 'SUBARU' 'SCION' 'LEXUS' 'MINI' 'VOLVO'
 'N/A']
Color object ['RED' 'SILVER' 'WHITE' 'BLUE' 'BEIGE' 'BLACK' 'GREEN' 'GREY' 'N/A' 'GOLD'
 'PURPLE' 'ORANGE' 'MAROON' 'YELLOW' 'OTHER' 'BROWN']
Transmission object ['AUTO' 'MANUAL' 'N/A']
WheelType object ['COVERS' 'ALLOY' 'N/A' 'SPECIAL']
VehOdo Int32 <IntegerArray>
[ 51099,  48542,  46318,  50413,  50199, 480444,  48433,  51062,  59825,
  49558,
 ...
  57444,  76391,  44622,  69941,  93744,  74407,  82563,  65399,  45234,
  66855]
Length: 28605, dtype: Int32
Nationality object ['AMERICAN' 'OTHER ASIAN' 'USA' 'TOP LINE ASIAN' '

,Auction,VehYear,Make,Color,Transmission,WheelType,VehOdo,Nationality,Size,MMRAcquisitionAuctionAveragePrice,...,MMRCurrentRetailCleanPrice,MMRCurrentRetailRatio,PRIMEUNIT,AUCGUART,VNST,VehBCost,IsOnlineSale,WarrantyCost,ForSale,IsBadBuy
0,OTHER,2008,DODGE,RED,AUTO,COVERS,51099,AMERICAN,MEDIUM,8566,...,12505,0.941783,NaN,N/A,NC,7800,False,920.0,True,False
1,OTHER,2008,DODGE,RED,AUTO,COVERS,48542,AMERICAN,MEDIUM,8566,...,10571,0.922618,NaN,N/A,NC,7800,False,834.0,True,False
2,OTHER,2008,CHRYSLER,SILVER,AUTO,COVERS,46318,AMERICAN,MEDIUM,8835,...,9932,0.935159,NaN,N/A,NC,7800,False,834.0,True,False
3,OTHER,2008,CHEVROLET,RED,AUTO,COVERS,50413,AMERICAN,COMPACT,7165,...,8739,0.931457,NaN,N/A,NC,6000,False,671.0,True,False
4,OTHER,2008,DODGE,SILVER,AUTO,COVERS,50199,AMERICAN,MEDIUM,8566,...,9908,0.906944,NaN,N/A,NC,7800,False,920.0,True,False
